In [26]:
import pandas as pd

In [27]:
df = pd.read_csv("IMDB Dataset.csv")

In [28]:
df.drop_duplicates(inplace=True)

### Pre Processing

In [29]:
df["review"] = df["review"].str.lower() # Convert to lowercase

In [30]:
# remove urls
import re
def remove_urls(text):
    text = re.sub(r"http\S+","",text) # (pattern,replacment,string)
    return text

df["review"] = df["review"].apply(remove_urls)

In [31]:
# remove Punctuations
import re
def remove_Punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text) # (pattern,replacment,string)  [excludes]
    return text

df["review"] = df["review"].apply(remove_Punctuations)

In [32]:
# remove Html tags
import re
def remove_Html(text):
    text = re.sub(r"<.*?>","",text) # (pattern,replacment,string)  [excludes]
    return text

df["review"] = df["review"].apply(remove_Html)

In [33]:
# removing Stopwords
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [34]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [35]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [36]:
from nltk.stem import PorterStemmer

In [37]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_tokens = ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [38]:
# encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [39]:
y = df["sentiment"]

In [41]:
# vectorization 

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

### Datasets and DataLoaders

In [43]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X,y , random_state=42, test_size= 0.2
)

In [44]:
import torch
from torch.utils.data import DataLoader,TensorDataset

In [45]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [46]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_5616\2931448922.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(y_train.values).float()


In [47]:
train_loader = DataLoader(train_set,shuffle=True,batch_size=64)
test_loader = DataLoader(test_set,shuffle=True,batch_size=64)

### Build RNN

In [50]:
import torch.nn as nn
import torch.optim as optim

In [51]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        # FC layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out ,_ = self.rnn(x,h0)

        out = self.fc(out[:,-1,:])
        return out

In [52]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training model

In [53]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze())
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.22588157653808594
epoch = 2/10 and loss = 0.2755386531352997
epoch = 3/10 and loss = 0.17890720069408417
epoch = 4/10 and loss = 0.2683817744255066
epoch = 5/10 and loss = 0.1453985720872879
epoch = 6/10 and loss = 0.26109251379966736
epoch = 7/10 and loss = 0.2086690217256546
epoch = 8/10 and loss = 0.205764502286911
epoch = 9/10 and loss = 0.3616786599159241
epoch = 10/10 and loss = 0.19991789758205414


In [54]:
# evaluate 

model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0

    for Xb,yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs  = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted ==yb).sum().item()
    print(f"accuracy = {correct_vals/total_vals*100}")

accuracy = 85.88282746798427
